In [56]:
%useLatestDescriptors

%use dataframe
%use kandy
%use ktor-client

In [57]:
val serviceKeyFilePath = "/Users/unchil/AndroidStudioProjects/OceanWaterInfo/collectionServer/src/main/resources/application.json"
val configData = DataRow.readJson(path=serviceKeyFilePath)

In [83]:
import java.time.LocalDateTime
import java.time.format.DateTimeFormatter

val formatter = DateTimeFormatter.ofPattern("yyyy-MM-dd HH:mm:ss")
val now = LocalDateTime.now()
val previousHour = now.minusHours(12)

val wtch_dt_start = previousHour.format(formatter)
val wtch_dt_end = now.format(formatter)

println("wtch_dt_start:${wtch_dt_start}, wtch_dt_end:${wtch_dt_end}")

wtch_dt_start:2026-07-21 09:34:34, wtch_dt_end:2026-07-21 21:34:34


In [84]:
import java.net.URLEncoder
import java.nio.charset.StandardCharsets

val url = "${configData.MOF_API.endPoint}/${configData.MOF_API.subPath}" +
        "?wtch_dt_start=${URLEncoder.encode(wtch_dt_start, StandardCharsets.UTF_8.toString())}" +
        "&wtch_dt_end=${URLEncoder.encode(wtch_dt_end, StandardCharsets.UTF_8.toString())}" +
        "&numOfRows=1000" +
        "&ServiceKey=${configData.MOF_API.apikey}"

In [85]:
@file:DependsOn("org.json:json:20250107")

In [86]:
import org.json.XML

fun loadData(path:String):DataFrame<*> {
    var requestPage = 1
    val rows = mutableListOf<DataFrame<*>>()

    try {
        do{
            val pagePath = "$path&pageNo=$requestPage"
            val response = http.get(pagePath)
            if (response.status.value == 200) {
                try {
                    XML.toJSONObject(response.bodyAsText()).let { jsonData ->
                        val df = DataFrame.readJson(jsonData.toString().byteInputStream())
                        val result = df.get("response").get("body").get("items").get("item")[0] as DataFrame<*>
                        requestPage += 1
                        rows.add(result)
                    }
                } catch(e: Exception) {
                    println(e.localizedMessage)
                    break
                }
            } else {
                println("${response.status.description}")
            }
        } while (requestPage < 500 )
    } catch(e:Exception ){
        println(e.localizedMessage)
    }
    return rows.concat()
}


In [87]:
val dfRaw = loadData(url)

dfRaw.describe()

Column not found: 'items'


name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
rtmWqDoxn,Number,1078,679,0,0.260000,37,5.903647,3.153307,0.250000,4.700000,5.910000,6.790000,19.907000
rtmWqChpla,Comparable<*>,1078,779,0,,89,null,null,null,null,null,null,null
rtmWqBgalgsQy,String,1078,1,0,,1078,null,null,,,,,
rtmWqWtchStaCd,String,1078,14,0,SEA5003,90,null,null,NEP1002,NEP3001,SEA2005,SEA5002,SEA7002
num,Int,1078,1078,0,1,1,539.500000,311.336099,1,269.916667,539.500000,809.083333,1078
rtmWqTu,Int,1078,143,0,4,101,29.928571,49.444617,0,5.000000,12.000000,27.000000,486
ph,Number,1078,182,0,7.510000,29,7.730928,0.360112,7.130000,7.480000,7.660000,7.940000,9.080000
rtmWqSlnty,Number,1078,1022,0,26.632999,4,22.028170,10.276752,0.020000,14.018000,26.632999,29.643000,34.032001
rtmWqCndctv,Float,1078,1047,0,39.435001,2,34.452099,15.359155,0.046000,23.147750,41.193001,44.919332,54.451000
rtmWqWtchDtlDt,String,1078,94,0,2026-07-21 09:40:00.0,14,null,null,2026-07-21 09:40:00.0,2026-07-21 12:30:00.0,2026-07-21 15:30:00.0,2026-07-21 18:30:00.0,2026-07-21 21:15:00.0


In [88]:
val df = dfRaw.parse().remove{ rtmWqBgalgsQy }.convert { rtmWqChpla }.with {
    val value = it.toString().trim()
    if (value.isNullOrBlank()) "0" else String.format("%.3f", value.toFloat())

}.convert { rtmWqDoxn and  rtmWqSlnty and rtmWqCndctv and rtmWtchWtem  and ph }.with { String.format("%.3f", it.toFloat()) }


df.schema()

rtmWqDoxn: String
rtmWqChpla: String
rtmWqWtchStaCd: String
num: Int
rtmWqTu: Int
ph: String
rtmWqSlnty: String
rtmWqCndctv: String
rtmWqWtchDtlDt: LocalDateTime
rtmWtchWtem: String

In [89]:
val newColunmNames = listOf( "용존산소", "클로로필", "관측정점코드", "순번", "탁도", "수소이온농도", "염분", "전기전도도", "일시", "수온")
val renamePairs = df.columnNames().zip(newColunmNames).toTypedArray()
val renamedDf = df.rename(*renamePairs)
renamedDf.columnNames()

[용존산소, 클로로필, 관측정점코드, 순번, 탁도, 수소이온농도, 염분, 전기전도도, 일시, 수온]

In [90]:
val removedDf = renamedDf.move { 순번 and 일시 }.toStart()

removedDf.head(5)

순번,일시,용존산소,클로로필,관측정점코드,탁도,수소이온농도,염분,전기전도도,수온
1,2026-07-21T09:40,3.170,0,SEA1005,25,7.290,2.567,4.810,26.000
2,2026-07-21T09:40,5.370,0.780,SEA1301,5,7.770,30.090,45.028,26.170
3,2026-07-21T09:40,5.620,9.080,SEA5003,15,7.640,32.829,48.972,23.820
4,2026-07-21T09:40,5.480,1.240,SEA5001,4,7.340,8.496,15.097,28.250
5,2026-07-21T09:40,7.780,2.650,NEP1002,8,7.860,0.460,0.933,29.380


In [91]:
removedDf
    .select{  일시 and 수온 and 관측정점코드   }
    .convert{수온}.toDouble()
    .plot{
        layout {
            title = "한국 해수 정보"
            size = 2600 to 600
        }
        x(일시) { axis.name = "관측일시"}
        y(수온) {axis.name ="수온 °C"}
        line{
            color(관측정점코드){
                //   scale = continuous(Color.GREEN..Color.RED)
                legend{
                    name = "관측정점코드"
                }
            }
        }
    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="L7cZJj" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 2600.0, 
 height: 600.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("L7cZJj");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"한국 해수 정보"
},
"mapping":{
},
"data":{
"일시":[1.7846268E12,1.7846268E12,1.7846268E12,1.7846268E12,1.7846268E12,1.7846268E12,1.7846268E12,1.7846268E12,1.7846268E12,1.7846268E12,1.7846268E12,1.7846268E12,1.7846268E12,1.7846268E12,1.7846271E12,1.7846271E12,1.7846271E12,1.7846271E12,1.7846271E12,1.7846271E12,1.7846271E12,1.7846271E12,1.7846271E12,1.7846277E12,1.7846277E12,1.7846277E12,1.7846277E12,1.7846277E12,1.7846277E12,1.7846277E12,1.7846277E12,1.7846277E12,1.7846277E12,1.7846277E12,1.7846277E12,1.7846277E12,1.784628E12,1.784628E12,1.784628E12,1.784628E12,1.784628E12,1.784628E12,1.784628E12,1.784628E12,1.784628E12,1.784628E12,1.7846286E12,1.7846286E12,1.7846286E12,1.7846286E12,1.7846286E12,1.7846286E12,1.7846286E12,1.7846286E12,1.7846286E12,1.7846286E12,1.7846286E12,1.7846286E12,1.7846286E12,1.7846286E12,1.7846289E12,1.7846289E12,1.7846289E12,1.7846289E12,1.7846289E12,1.7846289E12,1.7846289E12,1.7846289E12,1.7846289E12,1.7846289E12,1.7846289E12,1.7846295E12,1.7846295E12,1.7846295E12,1.7846295E12,1.7846295E12,1.7846295E12,1.7846295E12,1.7846295E12,1.7846295E12,1.7846295E12,1.7846295E12,1.7846295E12,1.7846295E12,1.7846298E12,1.7846298E12,1.7846298E12,1.7846298E12,1.7846298E12,1.7846298E12,1.7846298E12,1.7846298E12,1.7846298E12,1.7846304E12,1.7846304E12,1.7846304E12,1.7846304E12,1.7846304E12,1.7846304E12,1.7846304E12,1.7846304E12,1.7846304E12,1.7846304E12,1.7846304E12,1.7846304E12,1.7846304E12,1.7846307E12,1.7846307E12,1.7846307E12,1.7846307E12,1.7846307E12,1.7846307E12,1.7846307E12,1.7846307E12,1.7846307E12,1.7846307E12,1.7846313E12,1.7846313E12,1.7846313E12,1.7846313E12,1.7846313E12,1.7846313E12,1.7846313E12,1.7846313E12,1.7846313E12,1.7846313E12,1.7846313E12,1.7846313E12,1.7846313E12,1.7846316E12,1.7846316E12,1.7846316E12,1.7846316E12,1.7846316E12,1.7846316E12,1.7846316E12,1.7846316E12,1.7846322E12,1.7846322E12,1.7846322E12,1.7846322E12,1.7846322E12,1.7846322E12,1.7846322E12,1.7846322E12,1.7846322E12,1.7846322E12,1.7846322E12,1.7846322E12,1.7846325E12,1.7846325E12,1.7846325E12,1.7846325E12,1.7846325E12,1.7846325E12,1.7846325E12,1.7846325E12,1.7846331E12,1.7846331E12,1.7846331E12,1.7846331E12,1.7846331E12,1.7846331E12,1.7846331E12,1.7846331E12,1.7846331E12,1.7846331E12,1.7846331E12,1.7846331E12,1.7846331E12,1.7846334E12,1.7846334E12,1.7846334E12,1.7846334E12,1.7846334E12,1.7846334E12,1.7846334E12,1.7846334E12,1.7846334E12,1.784634E12,1.784634E12,1.784634E12,1.784634E12,1.784634E12,1.784634E12,1.784634E12,1.784634E12,1.784634E12,1.784634E12,1.784634E12,1.784634E12,1.7846343E12,1.7846343E12,1.7846343E12,1.7846343E12,1.7846343E12,1.7846343E12,1.7846343E12,1.7846343E12,1.7846343E12,1.7846349E12,1.7846349E12,1.7846349E12,1.7846349E12,1.7846349E12,1.7846349E12,1.7846349E12,1.7846349E12,1.7846349E12,1.7846349E12,1.7846349E12,1.7846349E12,1.7846349E12,1.7846349E12,1.7846352E12,1.7846352E12,1.7846352E12,1.7846352E12,1.7846352E12,1.7846352E12,1.7846352E12,1.7846352E12,1.7846352E12,1.7846352E12,1.7846352E12,1.7846358E12,1.7846358E12,1.7846358E12,1.7846358E12,1.7846358E12,1.7846358E12,1.7846358E12,1.7846358E12,